In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

df = pd.read_csv('../data/processed/spain_top50_with_runs.csv', parse_dates=['date'])
lifecycle = pd.read_csv(
    '../data/processed/lifecycle_table.csv',
    parse_dates=['entry_date', 'exit_date', 'peak_date', 'first_run_peak_date']
)

print(f"Day-by-day rows: {df.shape}")
print(f"Lifecycle rows: {lifecycle.shape}")

# Length of every run, in chart days
run_lengths = df.groupby(['song_key', 'artist_key', 'run_id']).size().rename('run_length')
print(f"\nTotal runs: {len(run_lengths)}")
print(run_lengths.describe())
print(run_lengths.quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

Day-by-day rows: (27750, 14)
Lifecycle rows: (575, 21)

Total runs: 1003
count    1003.000000
mean       27.666999
std        51.150231
min         1.000000
25%         2.000000
50%         6.000000
75%        24.000000
max       387.000000
Name: run_length, dtype: float64
0.10      1.0
0.25      2.0
0.50      6.0
0.75     24.0
0.90    100.0
Name: run_length, dtype: float64


In [3]:
df = df.sort_values(['song_key', 'artist_key', 'date'])

runs = (
    df.groupby(['song_key', 'artist_key', 'run_id'])
    .agg(run_length=('date', 'count'),
         best_position=('position', 'min'),
         entry_position=('position', 'first'))
    .reset_index()
)

runs['length_group'] = pd.cut(
    runs['run_length'],
    bins=[0, 2, 5, 14, 60, 1000],
    labels=['1-2 days', '3-5 days', '6-14 days', '15-60 days', '60+ days']
)

summary = runs.groupby('length_group', observed=True).agg(
    num_runs=('run_length', 'count'),
    median_best_position=('best_position', 'median'),
    share_never_above_41=('best_position', lambda s: (s >= 41).mean()),
    median_entry_position=('entry_position', 'median')
)
summary['share_never_above_41'] = summary['share_never_above_41'].map('{:.0%}'.format)
summary

,num_runs,median_best_position,share_never_above_41,median_entry_position
length_group,,,,
1-2 days,305,46.0,70%,47.0
3-5 days,192,42.0,52%,46.0
6-14 days,183,32.0,33%,44.0
15-60 days,176,25.0,8%,41.0
60+ days,147,7.0,0%,29.0


In [4]:
runs['is_fringe'] = (runs['run_length'] <= 5) & (runs['best_position'] >= 41)

print(f"Fringe runs: {runs['is_fringe'].sum()} of {len(runs)} ({runs['is_fringe'].mean():.0%})")

per_song = runs.groupby(['song_key', 'artist_key'])['is_fringe'].agg(['any', 'all'])
print(f"Songs with at least one fringe run: {per_song['any'].sum()} of {len(per_song)}")
print(f"Songs whose runs are ALL fringe: {per_song['all'].sum()}")
print()

# Re-test the 'peaks on entry day' finding, split by fringe vs not
first = runs[runs['run_id'] == 1].merge(
    lifecycle[['song_key', 'artist_key', 'entered_before_data', 'time_to_peak_first_run']],
    on=['song_key', 'artist_key']
)
first = first[~first['entered_before_data']]

for label, grp in first.groupby('is_fringe'):
    share = (grp['time_to_peak_first_run'] == 0).mean()
    name = 'fringe first run' if label else 'non-fringe first run'
    print(f"{name}: {len(grp)} songs, {share:.0%} peak on entry day")

Fringe runs: 314 of 1003 (31%)
Songs with at least one fringe run: 195 of 575
Songs whose runs are ALL fringe: 74

non-fringe first run: 422 songs, 52% peak on entry day
fringe first run: 103 songs, 86% peak on entry day


In [5]:
NEW_DAYS = 3        # first 3 days of a run are 'New Entry'
PEAK_TOL = 3        # within 3 positions of the run's best = 'Peak'
DECLINE_GAP = 10    # more than 10 positions below the best = 'Decline'

keys = ['song_key', 'artist_key', 'run_id']

helper_cols = ['best_position', 'is_fringe', 'day_in_run', 'in_peak_zone', 'first_peak_zone_date', 'stage']
df = df.drop(columns=[c for c in helper_cols if c in df.columns])

df = df.sort_values(keys + ['date']).reset_index(drop=True)
df = df.merge(runs[keys + ['best_position', 'is_fringe']], on=keys, how='left')

df['day_in_run'] = df.groupby(keys).cumcount() + 1
df['in_peak_zone'] = df['position'] <= df['best_position'] + PEAK_TOL

first_zone = df[df['in_peak_zone']].groupby(keys)['date'].min().rename('first_peak_zone_date').reset_index()
df = df.merge(first_zone, on=keys, how='left')

conditions = [
    df['is_fringe'],
    df['day_in_run'] <= NEW_DAYS,
    df['in_peak_zone'],
    df['date'] < df['first_peak_zone_date'],
    (df['position'] - df['best_position']) > DECLINE_GAP,
]
choices = ['Fringe', 'New Entry', 'Peak', 'Growth', 'Decline']
df['stage'] = np.select(conditions, choices, default='Mature')

order = ['New Entry', 'Growth', 'Peak', 'Mature', 'Decline', 'Fringe']
counts = df['stage'].value_counts().reindex(order)
print(pd.DataFrame({'song_days': counts, 'share': (counts / len(df)).map('{:.1%}'.format)}))

           song_days  share
stage                      
New Entry       1913   6.9%
Growth          2017   7.3%
Peak            5430  19.6%
Mature          6468  23.3%
Decline        11274  40.6%
Fringe           648   2.3%


In [6]:
# Stage mix (% of song-days) within each run-length group
check = df.merge(runs[keys + ['length_group']], on=keys, how='left')
mix = pd.crosstab(check['length_group'], check['stage'], normalize='index').reindex(columns=order)
print((mix * 100).round(0).astype(int))

# The corrected speed finding I promised earlier (non-fringe first runs only)
nf_first = first[~first['is_fringe']]
print(f"\nNon-fringe first runs peaking within 4 days: {(nf_first['time_to_peak_first_run'] <= 4).mean():.0%}")

stage         New Entry  Growth  Peak  Mature  Decline  Fringe
length_group                                                  
1-2 days             30       0     0       0        0      70
3-5 days             37       0     3       3        6      50
6-14 days            33       3    25      17       21       0
15-60 days           10      10    19      26       35       0
60+ days              2       7    20      24       46       0

Non-fringe first runs peaking within 4 days: 71%


In [7]:
# Position a week earlier within the same run (empty for the first 7 days of a run)
df['pos_week_ago'] = df.groupby(keys)['position'].shift(7)
still_sliding = (df['position'] > df['pos_week_ago']) | df['pos_week_ago'].isna()

conditions = [
    df['is_fringe'],
    df['day_in_run'] <= NEW_DAYS,
    df['in_peak_zone'],
    df['date'] < df['first_peak_zone_date'],
    ((df['position'] - df['best_position']) > DECLINE_GAP) & still_sliding,
]
df['stage_v2'] = np.select(conditions, choices, default='Mature')

# Overall comparison: old rule vs new rule (% of song-days)
compare = pd.DataFrame({
    'v1_share': df['stage'].value_counts(normalize=True).reindex(order),
    'v2_share': df['stage_v2'].value_counts(normalize=True).reindex(order),
})
print((compare * 100).round(1))

# Stage mix by run length under the new rule
check = df.merge(runs[keys + ['length_group']], on=keys, how='left')
mix = pd.crosstab(check['length_group'], check['stage_v2'], normalize='index').reindex(columns=order)
print()
print((mix * 100).round(0).astype(int))

           v1_share  v2_share
New Entry       6.9       6.9
Growth          7.3       7.3
Peak           19.6      19.6
Mature         23.3      34.2
Decline        40.6      29.8
Fringe          2.3       2.3

stage_v2      New Entry  Growth  Peak  Mature  Decline  Fringe
length_group                                                  
1-2 days             30       0     0       0        0      70
3-5 days             37       0     3       3        6      50
6-14 days            33       3    25      18       20       0
15-60 days           10      10    19      31       30       0
60+ days              2       7    20      38       32       0


In [8]:
if 'stage_v2' in df.columns:
    df['stage'] = df['stage_v2']
    df = df.drop(columns=['stage_v2', 'pos_week_ago'])

df.to_csv('../data/processed/spain_top50_staged.csv', index=False)

stage_summary = df['stage'].value_counts().reindex(order).rename('song_days').to_frame()
stage_summary['share_pct'] = (stage_summary['song_days'] / len(df) * 100).round(1)
stage_summary.to_csv('../outputs/tables/stage_distribution.csv')

print(df.shape)
print(stage_summary)

(27750, 20)
           song_days  share_pct
stage                          
New Entry       1913        6.9
Growth          2017        7.3
Peak            5430       19.6
Mature          9478       34.2
Decline         8264       29.8
Fringe           648        2.3
